# From E(t) to a donor–acceptor distance distribution

The analysis at the heart of the E(t) paper, rebuilt on the current API: take
the time-resolved transfer efficiency, and deconvolve it into a distribution of
donor–acceptor distances.

A donor at distance $R$ decays as $\exp[-t/\tau_0 (1 + (R_0/R)^6)]$, so a
mixture of distances gives

$$ f_{DA}(t) / f_{D0}(t) = \int p(R)\, e^{-t/\tau_0 (R_0/R)^6} \, dR $$

which is a linear problem in $p(R)$ — solved here with non-negative least
squares and a smoothness penalty. The old notebook used `mfm`'s
`calc_transfer_matrix` and `solve_nnls`; both are a few lines of NumPy/SciPy.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "quest_notebook.py").exists() else Path.cwd() / "notebooks"))
from quest_notebook import REPO, STRUCTURE, quick_project, fret_project

import numpy as np
import matplotlib.pyplot as plt

print("structure:", STRUCTURE.name)

In [ ]:
from scipy.optimize import nnls
from quest.project import DEFAULT_FORSTER_RADIUS
from quest.api import simulate_project

R0 = DEFAULT_FORSTER_RADIUS
TAU0 = 4.2

p = fret_project(donor=55, acceptor=132, r0=R0, n_photons=400_000, tau0=TAU0, random_seed=11)
result = simulate_project(p, save_outputs=False)
print("E =", round(result.fret_efficiency, 3))


In [ ]:
def transfer_matrix(t, r_axis, r0=R0, tau0=TAU0):
    """Column j is the FRET-induced decay of a pair at distance r_axis[j]."""
    k = (1.0 / tau0) * (r0 / r_axis[None, :]) ** 6
    return np.exp(-t[:, None] * k)

t = result.time
fd0 = result.donor_counts.astype(float)
fda = result.fret_counts.astype(float)

# Use the part of the decay with real statistics.
usable = (fd0 > 20.0) & (t < 20.0)
t_fit = t[usable]
et = fda[usable] / fd0[usable]

r_axis = np.linspace(20.0, 90.0, 96)
A = transfer_matrix(t_fit, r_axis)

# Second-difference smoothness penalty: without it NNLS answers with spikes.
penalty = 0.05 * np.sqrt(len(t_fit))
D = np.diff(np.eye(len(r_axis)), n=2, axis=0)
A_reg = np.vstack([A, penalty * D])
b_reg = np.concatenate([et, np.zeros(D.shape[0])])

p_r, residual = nnls(A_reg, b_reg)
p_r = p_r / p_r.sum()
mean_r = float((p_r * r_axis).sum())
print(f"recovered <R_DA> = {mean_r:.1f} A")

## Cross-check against the accessible volumes

The distance distribution came from the *decay*. The accessible volumes give an
independent estimate of the same quantity — the Förster-averaged distance
⟨R_DA⟩_E — so the two should agree to within a few Ångström.

In [ ]:
from quest.core import build_donor_from_project

model = build_donor_from_project(p)
model.update_all(verbose=False)
rda_e = model.rda_mean_e(n_samples=20000)
print(f"from the AV clouds : <R_DA>_E = {rda_e:.1f} A")
print(f"from the decay     : <R>      = {mean_r:.1f} A")
print(f"difference         : {abs(rda_e - mean_r):.1f} A")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(r_axis, p_r, lw=2)
ax.axvline(rda_e, color="k", ls="--", label=f"$\\langle R_{{DA}}\\rangle_E$ from AVs = {rda_e:.0f} A")
ax.axvline(mean_r, color="C3", ls=":", label=f"mean of p(R) = {mean_r:.0f} A")
ax.set_xlabel("$R_{DA}$ [A]"); ax.set_ylabel("p(R)")
ax.set_title("Distance distribution deconvolved from E(t)")
ax.legend(); plt.tight_layout()